# P02 · A real fMRI run: metadata, design, model, and statistical map

<!-- paper-first -->
### Research question

**Reading:** [PP01](../../curriculum/papers/processing.md#pp01), [PD01](../../curriculum/papers/design.md#pd01), [PB01](../../curriculum/papers/measurement.md#pb01). Review the assigned figure or result before starting the lesson.

**Question:** Which links between these raw teaching images and a biological claim remain unsupported by this limited GLM?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Learning format:** predict → ask AI for one short operation → run → inspect → deliberately break → explain. Use Goose with a local Ollama model, or ChatGPT as a tutor and snippet writer. You are responsible for deciding whether the transformation answers the research question. Read the answer guide only after making your own prediction.

**Time:** 4–6 hours. **Prerequisites:** processing fMRI sequence and research-design GLM/inference sequence. **Execution tier:** public-data download, explicitly excluded from offline CI. Run from this repository or its notebook subdirectories. The dataset is about 30 MB compressed; analysis requires additional memory. Raw and derived image files stay in ignored `data/` and `outputs/` folders.

Use the [UCL SPM auditory teaching dataset](https://www.fil.ion.ucl.ac.uk/spm/data/auditory/) and compare this exercise with the [Nilearn single-run GLM tutorial](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_single_subject_single_run.html). The source offers the data for personal education and evaluation; this repository downloads rather than redistributes it. Credit Geraint Rees, Karl Friston, and the FIL methods group.

The crucial discovery is in the BIDS sidecar: the supplied image has 84 volumes, TR is 7 seconds, and 12 volumes have already been discarded by the user. Use the supplied events relative to that delivered series. Do not remove another 12 volumes or shift events merely because a generic assistant suggests “discard initial scans.” Input metadata controls the decision.

This is a transparent GLM demonstration on the supplied raw teaching images. It does **not** implement motion realignment, distortion correction, anatomical registration, or nuisance regression with measured motion parameters. Smoothing and AR(1) noise modeling do not replace those steps. Treat the output as a learning result, not a completed research analysis. A required extension is to produce a documented preprocessing derivative with the specialist practicals, then repeat the model and compare QC and estimates.

The model predicts each voxel's observed series using an HRF-convolved listening regressor, low-frequency drift terms, and an intercept. AR(1) models residual temporal dependence; it is an assumption to check. A listening effect estimate describes amplitude on the model's chosen signal scale. A z map combines estimated effect and uncertainty. FDR thresholding controls a stated error criterion under its assumptions; it does not turn this one person into population evidence or give the probability that each retained voxel is truly active.

Ask AI: **“Inspect the sidecar and event timing before constructing the design. Name every modeled and unmodeled nuisance source. Keep effect, z statistic, and thresholded display distinct. Explain the inferential family and why this one-run result cannot support a population or diagnostic claim.”**


### Load the public input and verify the delivered time axis
This is the only core project that can download participant imaging data; inspect its educational terms first.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, nibabel as nib
from nilearn.datasets import fetch_spm_auditory
root = Path.cwd().resolve()
while not (root/'notebooks').is_dir() and root != root.parent:
    root = root.parent
assert (root/'notebooks').is_dir(), 'Run inside the course repository'
data = fetch_spm_auditory(data_dir=root/'data',verbose=0)
func = data.func[0]
image = nib.load(func)
events = pd.read_csv(data.events,sep='\t')
sidecar = json.loads((Path(func).parents[2]/'task-auditory_bold.json').read_text())
tr = float(sidecar['RepetitionTime'])
print('shape:',image.shape,'TR:',tr,'already discarded:',sidecar['NumberOfVolumesDiscardedByUser'])
print(events.to_string(index=False))
assert image.shape[-1] == 84 and tr == data.t_r == 7.0
assert (events.onset >= 0).all() and (events.onset+events.duration <= image.shape[-1]*tr).all()

### Fit and inspect the design
No measured motion confounds are available here. Keep that omission visible in the methods.

In [ ]:
from nilearn.glm.first_level import FirstLevelModel
from nilearn.plotting import plot_design_matrix
import matplotlib.pyplot as plt
model = FirstLevelModel(t_r=tr,noise_model='ar1',hrf_model='spm',drift_model='cosine',high_pass=0.01,standardize=False,signal_scaling=0,smoothing_fwhm=6,minimize_memory=False)
model.fit(func,events=events)
design = model.design_matrices_[0]
assert len(design) == image.shape[-1] and 'listening' in design
assert np.linalg.matrix_rank(design.to_numpy()) == design.shape[1]
print('Design shape:',design.shape,'columns:',list(design.columns))
print('Design condition number:',np.linalg.cond(design.to_numpy()))
plot_design_matrix(design); plt.show()

### Distinguish effect, uncertainty, and display
The threshold uses the fitted mask as the family and no cluster-size inference.

In [ ]:
from nilearn.glm import threshold_stats_img
from nilearn.plotting import plot_stat_map
zmap = model.compute_contrast('listening',output_type='z_score')
effect = model.compute_contrast('listening',output_type='effect_size')
thresholded, zcut = threshold_stats_img(zmap,mask_img=model.masker_.mask_img_,alpha=0.05,height_control='fdr',cluster_threshold=0,two_sided=True)
inside = model.masker_.mask_img_.get_fdata().astype(bool)
z = zmap.get_fdata()[inside]
assert np.isfinite(z).all() and zmap.shape == effect.shape
summary = {'n_scans':image.shape[-1],'TR_seconds':tr,'n_mask_voxels':int(inside.sum()),'design_columns':list(design.columns),'max_abs_z':float(abs(z).max()),'two_sided_FDR_0.05_z':float(zcut),'retained_voxels':int(np.count_nonzero(thresholded.get_fdata())),'scope':'raw-image teaching GLM; no motion/distortion correction; no population inference'}
print(json.dumps(summary,indent=2))
plot_stat_map(thresholded,bg_img=data.anat,title='Teaching GLM: two-sided voxel FDR, raw input',display_mode='z',cut_coords=[0,20,40]); plt.show()
output = root/'outputs'/'P02'; output.mkdir(parents=True,exist_ok=True)
(output/'analysis_summary.json').write_text(json.dumps(summary,indent=2))

### Residual check
A fitted map is not the end. Compare residual lag-one dependence with the model assumption.

In [ ]:
residuals = model.residuals[0].get_fdata()[inside]
a,b = residuals[:,:-1],residuals[:,1:]
a = a-a.mean(axis=1,keepdims=True); b = b-b.mean(axis=1,keepdims=True)
denominator = np.sqrt((a*a).sum(axis=1)*(b*b).sum(axis=1))
lag1 = np.divide((a*b).sum(axis=1),denominator,out=np.zeros(len(a)),where=denominator>0)
print('Median raw residual lag-one correlation:',float(np.median(lag1)))
print('These are raw residuals, not evidence that whitened residuals are independent.')
plt.hist(lag1,bins=40); plt.xlabel('Raw residual lag-one correlation'); plt.ylabel('Voxels'); plt.show()
assert np.isfinite(lag1).all()

## Explain without AI

Submit the metadata check, design plot, mask inspection, effect/z distinction, residual diagnostic, and a list of unmodeled nuisance sources. Compare AR(1) with OLS and 6 mm with no smoothing as **labeled sensitivity analyses**, not opportunities to choose the prettiest result. High-pass 0.01 Hz lies near the 84-second task cycle; explain why drift removal can interact with task information. Then complete the upstream preprocessing practical and document every extra operation before claiming a research pipeline.

<details><summary>Answer guide</summary>The delivered 84-volume series already excludes 12 acquired volumes. Its event table must stay synchronized with it. Signal scaling uses each voxel's mean as a baseline; standardize=False does not disable that separate setting. A two-sided voxel FDR threshold is not a cluster-level test. Raw-image motion or distortion effects can survive modeling and invalidate biological interpretation. One participant cannot establish a population effect.</details>

**Submission:** record the input, operation, parameters, output, one preserved property, one lost property, and the evidence that would make you reject the result. Include the prompt and any corrections you made to the generated code. A saved answer is not evidence of understanding until you can defend it orally.


### Return to the research question

Revisit [PP01](../../curriculum/papers/processing.md#pp01), [PD01](../../curriculum/papers/design.md#pd01), [PB01](../../curriculum/papers/measurement.md#pb01) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
